# 2차시 실습 — 요구사항 명확화 · 문제 구조화 · 데이터 준비 (I)

**인공지능플랫폼 실습/설계 · 2026학년도 2학기**

---

## 이 실습의 목적

머신러닝 시스템 설계 7단계 중 **1~3단계**를 HR 이직 예측 사례에 적용한다.

| 단계 | 이름 | 이 노트북에서 하는 일 |
|---|---|---|
| 1 | 요구사항 명확화 | 모호한 요청을 6가지 축의 질문으로 좁혀 **요구사항 명세 카드** 작성 |
| 2 | ML 작업으로 문제 구조화 | ML 목표·입출력·학습유형을 정하고 **ML 문제 정의 카드** 작성 |
| 3 | 데이터 준비 (I) | 데이터 적재·스키마 검증·EDA 로 **데이터 카드** 작성 |

> **중요** — 이 실습의 결과물은 그래프가 아니라 **세 장의 설계 카드**다.
> 3·4차시 실습이 이 카드들을 이어받아 진행된다.

---

## 사용 데이터

`http://joy-bae.github.io/HR-Employee-Attrition.csv` — 직원 인사기록 1,470건 × 20개 속성

| 속성 | 의미 | 속성 | 의미 |
|---|---|---|---|
| Age | 나이 | OverTime | 초과 근무 여부 (Yes/No) |
| **Attrition** | **이직 여부 (목표 변수)** | p_SalaryHike | 급여 인상 비율(%) |
| Department | 근무 부서 | Performance | 성과 평가 등급 (1~4) |
| Dist | 집~회사 거리 | RelationshipSatisfaction | 동료 관계 만족도 (1~4) |
| Education | 학력 (1~5) | StockOption | 주식 옵션 수준 (0~3) |
| Env_Satisfaction | 환경 만족도 (1~4) | WorkLifeBalance | 일-생활 균형 (1~4) |
| Gender | 성별 | YearsAtCompany | 현 회사 근속 연수 |
| JobLevel | 직급 | YearsSinceLastPromotion | 마지막 승진 이후 경과 연수 |
| JobSatisfaction | 직업 만족도 (1~4) | Income_M | 월급 |
| MaritalStatus | 결혼 여부 | Companies | 이전 근무 회사 수 |


In [ ]:
# ============================================================
# 0. 환경 설정 (Colab 에서 런타임을 다시 연결할 때마다 실행)
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 한글 폰트 설정 (Colab) --------------------------------
import matplotlib.font_manager as fm, os, subprocess

def setup_korean_font():
    path = "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"
    if not os.path.exists(path):
        try:
            subprocess.run(["apt-get", "install", "-y", "fonts-nanum"],
                           check=False, capture_output=True)
        except Exception:
            pass
    if os.path.exists(path):
        fm.fontManager.addfont(path)
        plt.rcParams["font.family"] = "NanumBarunGothic"
    plt.rc("axes", unicode_minus=False)

setup_korean_font()
sns.set_theme(style="whitegrid", font=plt.rcParams["font.family"][0]
              if isinstance(plt.rcParams["font.family"], list) else plt.rcParams["font.family"])
plt.rc("axes", unicode_minus=False)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

RANDOM_STATE = 2026
DATA_URL = "http://joy-bae.github.io/HR-Employee-Attrition.csv"
print("환경 설정 완료 / pandas", pd.__version__)

---
# STEP 1 · 요구사항 명확화

## 상황

> OOO사 인사팀: **"요즘 직원들이 너무 많이 나갑니다. 인사기록 데이터가 있으니 AI로 어떻게 좀 해주세요."**

이것은 요구사항이 아니라 **불편함의 호소**다. 설계자는 여기서부터 질문을 시작한다.

## 실습 1-1. 명확화 질문 만들기

아래 6가지 축에 대해, 인사팀에게 **실제로 물어야 할 질문**을 각각 1개 이상 적어라.
(강의자료의 예시를 그대로 옮기지 말고, 이 사례에 맞게 구체적으로 쓸 것)


### [작성] 명확화 질문

아래 표의 빈칸을 채워라.

| 축 | 내가 물어볼 질문 | 인사팀의 답(가정) |
|---|---|---|
| 비즈니스 목표 |  |  |
| 피처 |  |  |
| 데이터 |  |  |
| 제약 |  |  |
| 시스템 규모 |  |  |
| 기대 성능 |  |  |


## 실습 1-2. 요구사항 명세 카드 (산출물 ①)

위 질문의 답을 정리해 아래 카드를 완성하라.
**"하지 않을 것"** 항목을 반드시 채울 것 — 시스템의 경계를 명시하지 않으면 예측 결과가 원래 의도와 다른 용도로 쓰이는 것을 막을 수 없다.


### [산출물 ①] 요구사항 명세 카드

- **문제 이름** :
- **요청자 / 사용자** :
- **비즈니스 목표** :
- **성공의 정의** :
- **사용 가능 데이터** :
- **제약** :
- **규모** :
- **성능 우선순위** :
- **하지 않을 것** :

**합의문 한 줄**
>


---
# STEP 2 · 머신러닝 작업으로 문제 구조화

## 실습 2-1. 머신러닝이 필요한가?

아래 네 조건을 이 사례에 대해 판정하라.

| 조건 | 판정 (O/X) | 근거 |
|---|---|---|
| ① 데이터에 패턴이 존재하는가 | | |
| ② 규칙으로 명시하기 어려운가 | | |
| ③ 라벨이 붙은 데이터가 있는가 | | |
| ④ 반복적으로 필요한 판단인가 | | |


### [작성] 판정 결과

| 조건 | 판정 | 근거 |
|---|---|---|
| ① 패턴 존재 |  |  |
| ② 규칙화 어려움 |  |  |
| ③ 라벨 존재 |  |  |
| ④ 반복 필요 |  |  |

**결론**:


## 실습 2-2. ML 문제 정의 카드 (산출물 ②)

비즈니스 목표를 머신러닝 목표로 **번역**하라.
머신러닝 목표에는 반드시 **예측 대상(무엇을)** · **예측 단위(누구마다)** · **예측 시점(언제)** 이 들어가야 한다.


### [산출물 ②] ML 문제 정의 카드

- **ML 목표** :
- **학습 유형** :
- **예측 단위** :
- **입력 X** :
- **출력 y** :
- **라벨 정의** :
- **사용 금지 피처** :
- **주 지표** :
- **기준선** :

**왜 회귀·군집화가 아닌가** (선택하지 않은 이유를 각각 한 문장으로)
- 회귀:
- 군집화:
- 다중 분류:


---
# STEP 3 · 데이터 준비 (I)

## 3-A. 데이터 적재와 스키마 검증

1차시 그림 1.1 의 머신러닝 시스템에는 `Data Collection` 옆에 **`Data Verification`** 블록이 따로 있었다.
학습을 시작하기 전에 데이터가 **우리가 기대한 계약(contract)** 을 지키는지 코드로 확인한다.

> 여기서 만드는 `validate()` 함수는 **4차시 배포·모니터링에서 그대로 재사용**한다.


In [ ]:
# 데이터 적재
df=
print("shape:", df.shape)


### 실습 3-1. 스키마 검증 함수 구현

`EXPECTED_SCHEMA` 를 참고해 `validate(df, schema)` 함수를 완성하라.

- 규칙이 `("num", lo, hi)` 이면 → 값이 `[lo, hi]` 범위를 벗어난 건수를 센다
- 규칙이 `("cat", {허용값})` 이면 → 허용값 집합에 없는 값이 있는지 확인한다
- 컬럼 자체가 없으면 `[누락]` 항목을 기록한다
- 문제가 없으면 빈 리스트를 반환한다


In [ ]:
EXPECTED_SCHEMA = {
    "Age":                      ("num", 18, 65),
    "Attrition":                ("num", 0, 1),
    "Department":               ("cat", {"Sales", "Research & Development", "Human Resources"}),
    "Dist":                     ("num", 1, 30),
    "Education":                ("num", 1, 5),
    "Env_Satisfaction":         ("num", 1, 4),
    "Gender":                   ("cat", {"Male", "Female"}),
    "JobLevel":                 ("num", 1, 5),
    "JobSatisfaction":          ("num", 1, 4),
    "MaritalStatus":            ("cat", {"Single", "Married", "Divorced"}),
    "Income_M":                 ("num", 1000, 20000),
    "Companies":                ("num", 0, 10),
    "OverTime":                 ("cat", {"Yes", "No"}),
    "p_SalaryHike":             ("num", 10, 25),
    "Performance":              ("num", 1, 4),
    "RelationshipSatisfaction": ("num", 1, 4),
    "StockOption":              ("num", 0, 3),
    "WorkLifeBalance":          ("num", 1, 4),
    "YearsAtCompany":           ("num", 0, 45),
    "YearsSinceLastPromotion":  ("num", 0, 20),
}


def validate(data, schema):
    """데이터가 기대한 스키마를 지키는지 확인하고 문제 목록을 반환한다."""
    issues = []
    for col, rule in schema.items():
        # TODO 1) 컬럼이 없으면 "[누락] ..." 메시지를 issues 에 추가하고 continue
        # TODO 2) rule[0] == "cat" 이면 허용값 집합(rule[1])에 없는 값이 있는지 확인
        # TODO 3) rule[0] == "num" 이면 [rule[1], rule[2]] 범위를 벗어난 건수를 확인
        # TODO 4) 결측치가 있으면 "[결측] ..." 메시지를 추가
        pass
    return issues


issues = validate(df, EXPECTED_SCHEMA)
print("검증 결과:", "이상 없음 " if not issues else "")
for i in issues:
    print(" -", i)

In [ ]:
# ─────────────────────────────────────────────────────────────
#  구제 셀 — validate() 스키마 검증 함수
#  위 TODO 를 완성했다면 이 셀은 그냥 실행하고 지나가면 된다 (아무 일도 일어나지 않는다).
#  막혔다면 USE_RESCUE 를 True 로 바꾸고 실행한 뒤 다음 절로 진행한다.
#  ※ 제출 시 True 로 두어도 감점하지 않는다. 다만 어디서 막혔는지는 기록해 둘 것.
# ─────────────────────────────────────────────────────────────
USE_RESCUE = False

if USE_RESCUE:
    def validate(data, schema):
        issues = []
        for col, rule in schema.items():
            if col not in data.columns:
                issues.append(f"[누락] 컬럼 '{col}' 이 없습니다")
                continue
            if rule[0] == "cat":
                unknown = set(data[col].dropna().unique()) - rule[1]
                if unknown:
                    issues.append(f"[값] {col}: 예상 밖의 값 {unknown}")
            else:
                lo, hi = rule[1], rule[2]
                if not pd.api.types.is_numeric_dtype(data[col]):
                    issues.append(f"[타입] {col}: 수치형이어야 하는데 {data[col].dtype} 입니다")
                    continue
                n_bad = int(((data[col] < lo) | (data[col] > hi)).sum())
                if n_bad:
                    issues.append(f"[범위] {col}: [{lo}, {hi}] 범위를 벗어난 값 {n_bad}건")
            n_na = int(data[col].isna().sum())
            if n_na:
                issues.append(f"[결측] {col}: 결측치 {n_na}건")
        return issues

    issues = validate(df, EXPECTED_SCHEMA)
    print("검증 결과:", "이상 없음" if not issues else issues)
    print('구제 코드를 사용했습니다 — validate() 스키마 검증 함수')

### 실습 3-2. 검증 함수가 정말 동작하는지 확인하기

검증 함수는 **문제가 있는 데이터를 넣었을 때 실제로 잡아내는지**를 확인해야 의미가 있다.
일부러 오염시킨 데이터를 만들어 넣어 보자.


In [ ]:
# 일부러 망가뜨린 데이터로 검증 함수를 시험한다 (운영 데이터에서 실제로 일어나는 일들)
broken = df.copy()
broken.loc[0, "Age"] = 150                      # 범위 이탈
broken.loc[1, "Department"] = "Marketing"       # 학습 때 없던 새 부서
broken.loc[2, "Income_M"] = np.nan              # 결측 발생
broken = broken.drop(columns=["StockOption"])   # 컬럼 삭제

for i in validate(broken, EXPECTED_SCHEMA):
    print(" -", i)

---
## 3-B. 기술통계와 속성 유형 분류

속성이 **수치형인지 범주형인지, 순서가 있는지 없는지**에 따라 3차시의 인코딩·스케일링 방식이 결정된다.

> 주의 — **숫자로 저장되어 있다고 수치형이 아니다.**
> `Education = 5` 가 `Education = 1` 보다 "5배"인 것은 아니다. 순서형 범주는 순서만 의미가 있다.


In [ ]:
df.info()

In [ ]:
# 수치형 기술통계
df.describe().T

In [ ]:
# 범주형 기술통계
df.describe(include="object").T

In [ ]:
# 범주형 데이터의 범주별 빈도수 확인 코드


### 실습 3-3. 속성 유형 분류표 만들기

아래 4가지 유형으로 20개 컬럼을 모두 분류하는 딕셔너리를 완성하고, 표로 출력하라.

- `num_cont` 수치형·연속형 / `num_disc` 수치형·이산형
- `cat_ord` 범주형·순서형 / `cat_nom` 범주형·명목형


In [ ]:
COLUMN_TYPES = {
    "num_cont": [],   # TODO: 수치형·연속형 컬럼
    "num_disc": [],   # TODO: 수치형·이산형 컬럼
    "cat_ord":  [],   # TODO: 범주형·순서형 컬럼 (정수로 저장된 만족도·등급 등)
    "cat_nom":  [],   # TODO: 범주형·명목형 컬럼 (순서 없는 문자열)
}
TARGET = "Attrition"

# 분류 누락 확인
classified = sum(COLUMN_TYPES.values(), []) + [TARGET]
missing = set(df.columns) - set(classified)
extra = set(classified) - set(df.columns)
print("분류되지 않은 컬럼:", missing if missing else "없음 ")
print("데이터에 없는 컬럼:", extra if extra else "없음 ")

label = {"num_cont": "수치형·연속형", "num_disc": "수치형·이산형",
         "cat_ord": "범주형·순서형", "cat_nom": "범주형·명목형"}
plan = {"num_cont": "스케일링 (왜도 크면 로그/로버스트 검토)",
        "num_disc": "스케일링 또는 버킷팅(이산화)",
        "cat_ord":  "정수 그대로 사용 (순서 의미 보존)",
        "cat_nom":  "원-핫 인코딩 (2범주는 라벨 인코딩)"}

type_table = pd.DataFrame(
    [{"컬럼": c, "유형": label[k], "고유값 수": df[c].nunique(),
      "3차시 처리 계획": plan[k]}
     for k, cols in COLUMN_TYPES.items() for c in cols]
)
type_table

In [ ]:
# ─────────────────────────────────────────────────────────────
#  구제 셀 — COLUMN_TYPES 속성 유형 분류
#  위 TODO 를 완성했다면 이 셀은 그냥 실행하고 지나가면 된다 (아무 일도 일어나지 않는다).
#  막혔다면 USE_RESCUE 를 True 로 바꾸고 실행한 뒤 다음 절로 진행한다.
#  ※ 제출 시 True 로 두어도 감점하지 않는다. 다만 어디서 막혔는지는 기록해 둘 것.
# ─────────────────────────────────────────────────────────────
USE_RESCUE = False

if USE_RESCUE:
    COLUMN_TYPES = {
        "num_cont": ["Income_M", "p_SalaryHike"],
        "num_disc": ["Age", "Dist", "Companies", "YearsAtCompany", "YearsSinceLastPromotion"],
        "cat_ord":  ["Education", "JobLevel", "JobSatisfaction", "Env_Satisfaction",
                     "RelationshipSatisfaction", "Performance", "StockOption", "WorkLifeBalance"],
        "cat_nom":  ["Department", "Gender", "MaritalStatus", "OverTime"],
    }
    TARGET = "Attrition"
    print("분류 완료 — 총", len(sum(COLUMN_TYPES.values(), [])) + 1, "개 컬럼")
    print('⚠ 구제 코드를 사용했습니다 — COLUMN_TYPES 속성 유형 분류')

---
## 3-C. 탐색적 데이터 분석 (EDA)

**EDA의 결론은 그래프가 아니라 문장이다.**
목표는 아래 **형태**의 문장 5개를 직접 찾아내는 것이다.

> "◯◯한 직원의 퇴사율이 △△%로, ◻◻한 직원(□□%)보다 약 ☆배 높다."

숫자가 들어가지 않은 문장("초과근무가 영향을 주는 것 같다")은 결론이 아니다.
여기서 찾은 문장들이 3차시의 피처 엔지니어링과 4차시의 모델 해석에 그대로 쓰인다.

**시작하기 전에 각자 예상해 보라** — 20개 속성 중 퇴사와 가장 관련이 클 것 같은 3개는 무엇인가?
EDA가 끝난 뒤 예상이 맞았는지 대조한다.


### 실습 3-4. 시각화 (1) — 라벨 분포와 범주별 퇴사율

In [ ]:
# TODO: 1행 4열의 subplot 을 만들어 아래를 그려라
#   [0] Attrition 의 countplot (x축 라벨을 '재직(0)', '퇴사(1)' 로 바꿀 것)
#   [1] Department 별 평균 퇴사율 막대그래프
#   [2] OverTime  별 평균 퇴사율 막대그래프
#   [3] MaritalStatus 별 평균 퇴사율 막대그래프
#   힌트: df.groupby(col)["Attrition"].mean() 이 곧 그 집단의 퇴사율이다
#   힌트: 전체 평균 퇴사율을 axhline 으로 그려 두면 비교가 쉽다


### 실습 3-5. 시각화 (2) — 수치형 속성의 분포 차이

In [ ]:
# TODO: Age, Income_M, YearsAtCompany, Dist 에 대해
#       이직 여부(Attrition)별 boxplot 을 1행 4열로 그려라.
#       그리고 groupby("Attrition") 로 네 컬럼의 평균을 비교 출력하라.


### 실습 3-6. 시각화 (3) — 속성 간 상관관계와 결합 분포

In [ ]:
# TODO: df.corr(numeric_only=True) 로 상관행렬을 구해 heatmap 으로 그려라.
#       이어서 Attrition 과의 상관계수를 절댓값 기준 내림차순으로 출력하라.
#       힌트: corr["Attrition"].drop("Attrition").sort_values(key=abs, ascending=False)


In [ ]:
sns.jointplot(data=df, x="Age", y="Income_M", hue="Attrition",
              palette=["#5B7CB2", "#A5093B"], height=5.5, alpha=0.6)
plt.suptitle("나이 × 월급 결합 분포", y=1.02)
plt.show()

---
## 3-D. 클래스 불균형 · 데이터 품질 · 민감 속성

이 절에서 확인할 수치 하나가 3·4차시의 모델 훈련 방식과 평가 지표 선택을 바꾸게 된다.
무엇이 그 수치일지 생각하면서 아래를 실행해 보라.


### 실습 3-7. 클래스 불균형과 기준선(baseline) 확인

STEP 2 에서 미뤄 두었던 **기준선 정확도**를 실제 데이터로 계산한다.


In [ ]:
# TODO: Attrition 의 클래스별 건수와 비율을 구하고,
#       '다수 클래스로만 예측했을 때의 정확도(= 기준선)' 를 출력하라.
#       그 기준선 모델의 재현율(Recall)은 얼마인가? 왜 그런지 생각해 보라.


### 실습 3-8. 데이터 품질 점검 — 결측 · 중복 · 이상치

처리 방법은 3차시에서 다루고, 오늘은 **무엇이 문제인지 발견하고 기록**한다.


In [ ]:
# TODO: 아래 세 가지를 확인해 출력하라.
#   1) 컬럼별 결측치 개수
#   2) 완전 중복 행의 개수
#   3) 수치형 컬럼별 IQR 기준 이상치 후보 개수와 왜도(skew)
#      IQR = Q3 - Q1, 하한 = Q1 - 1.5*IQR, 상한 = Q3 + 1.5*IQR


> **"이상치"와 "드문 정상값"은 다르다.**
> 월급이 매우 높은 임원은 이상치가 아니라 드문 정상값이다. 통계적 기준만으로 삭제하면 실제 신호를 지우게 된다.
> 아래에서 Income_M 의 상위 값이 실제로 무엇인지 확인해 보자.


In [ ]:
# Income_M 상위 이상치가 정말 '오류'인지 도메인 관점에서 확인
q1, q3 = df["Income_M"].quantile([0.25, 0.75])
hi = q3 + 1.5 * (q3 - q1)
outliers = df[df["Income_M"] > hi]
print(f"Income_M 상한 {hi:,.0f} 초과: {len(outliers)}건")
print("\n이상치 그룹의 직급(JobLevel) 분포:")
print(outliers["JobLevel"].value_counts().sort_index())
print("\n전체의 직급 분포:")
print(df["JobLevel"].value_counts().sort_index())
print("\n→ 고소득자는 대부분 상위 직급이다. 오류가 아니라 '드문 정상값' → 제거하지 않는다.")

### 실습 3-9. 민감 속성과 편향 점검

인사 데이터이므로 **개인정보와 편향**은 반드시 다뤄야 한다.
민감 속성(Gender, Age, MaritalStatus)에 대해 라벨 분포를 확인하고, 5차시… 가 아니라 **4차시 공정성 평가**의 기준선을 만들어 둔다.


In [ ]:
SENSITIVE = ["Gender", "MaritalStatus"]

# TODO 1) df 의 사본(df_sens)을 만들고, Age 를 [17,29,39,49,65] 구간으로 잘라
#         'Age_band' 컬럼을 추가하라 (pd.cut) — 원본 df 는 그대로 둘 것
# TODO 2) Gender, MaritalStatus, Age_band 각 그룹별로
#         인원 수 / 전체 대비 비중 / 실제 퇴사율 을 표로 출력하라
# → 이 표는 4차시 공정성 평가에서 '모델 예측'과 비교할 기준이 된다


> **민감 속성을 지우는 것은 해결책이 아니다.**
> Gender 컬럼을 제거해도 직무·부서·근속 패턴이 성별을 대리(proxy)한다.
> 우리가 할 수 있는 유일한 방법은 **측정하고 기록하는 것**이며, 그 결과는 4차시에서 모델 예측과 대조한다.


---
# 산출물 ③ 데이터 카드 (Data Card)

아래 셀을 실행해 데이터 카드를 자동 생성하고, 마지막의 **발견 사항 5문장**을 직접 작성하라.


In [ ]:
def make_data_card(data, types, target):
    """데이터 카드를 문자열로 생성한다."""
    lines = []
    lines.append("## 데이터 카드 (Data Card)\n")
    # TODO: 아래 항목을 채워 문자열로 만들어라
    #   - 출처 / 스냅샷
    #   - 규모 (행 × 열, 결측 건수, 중복 건수)
    #   - 라벨 분포와 소수 클래스 비율
    #   - 기준선 정확도
    #   - 유형별 컬럼 목록 (COLUMN_TYPES 활용)
    #   - 민감 속성
    return "\n".join(lines)


card = make_data_card(df, COLUMN_TYPES, TARGET)
print(card)

with open("data_card.md", "w", encoding="utf-8") as f:
    f.write(card)
print("\n→ data_card.md 저장 완료")

### [작성] EDA 결론 5문장 (산출물 ③의 '발견 사항')

실행 결과를 근거로, **숫자를 포함한 문장** 5개를 작성하라.
(예: "초과근무를 하는 직원의 퇴사율이 30.5%로 전체 평균의 약 2배")

1.
2.
3.
4.
5.

### [작성] 확인이 필요한 사항 (도메인 질문)

인사팀에 확인해야 할 항목을 2~3개 적어라.

-
-
-


---
# 제출

1. 모든 TODO 셀을 채우고 **런타임 → 모두 실행**이 오류 없이 끝나는지 확인한다.
2. 산출물 ①②③ (요구사항 명세 카드 · ML 문제 정의 카드 · 데이터 카드 + EDA 결론 5문장)이 모두 작성되었는지 확인한다.


## 다음 차시 (3차시) 예고

오늘 만든 **데이터 카드의 속성 유형 분류**를 실제 변환 코드로 바꾼다.
- 결측·이상치·스케일링·이산화·인코딩을 하나의 **Pipeline** 으로 묶는다
- 데이터 누수(leakage)를 막는 올바른 순서를 배운다
- 오늘 계산한 **기준선 정확도 83.9%** 를 실제로 넘어서는 모델을 만든다
